# IKEA Gifting – Customer Analytics & RFM Segmentation

**Workflow:** PostgreSQL data cleaning → Python/Pandas RFM analysis → Power BI.

This notebook starts from the SQL-cleaned CSV exported from `retail.retail_cleaned`. It calculates customer-level RFM metrics, scores customers, assigns segments, flags 90-day inactivity, merges customer attributes back into transactions, and exports a Power BI-ready file.

**Before running:** export the SQL table to `data/processed/retail_cleaned.csv`. Review the analysis filters and reference date below; record your final decisions in the README/report.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "retail_cleaned.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "retail_rfm_final.csv"

# Set this to the analysis snapshot date used in your project.
# Use a fixed date for reproducible results and document it in your report.
ANALYSIS_DATE = None  # Example: pd.Timestamp("2011-12-10")
CHURN_DAYS = 90

# Review these choices before analysis:
EXCLUDE_MISSING_CUSTOMER_ID = True
EXCLUDE_NONPOSITIVE_QUANTITY_FROM_RFM = True
EXCLUDE_NONPOSITIVE_PRICE_FROM_RFM = True


## 2. Load and inspect SQL-cleaned data

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_FILE}. Export retail.retail_cleaned from PostgreSQL to this path."
    )

df = pd.read_csv(INPUT_FILE)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

required = {"invoice", "stockcode", "description", "quantity",
            "invoice_date", "price", "customer_id", "country"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
for col in ["quantity", "price", "customer_id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows, columns:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing_values"))
display(df.describe(include="all").T)


## 3. Prepare the transaction data

The BRD defines Monetary as customer revenue over the analysis period and Frequency as distinct purchase transactions. Review cancellation / return treatment carefully. This starter workflow excludes non-positive quantities and prices from the RFM purchase calculations when the corresponding configuration flags are enabled. Keep the original SQL-cleaned table unchanged so the treatment remains auditable.

In [ ]:
analysis = df.copy()

analysis = analysis.dropna(subset=["invoice_date", "invoice", "price", "quantity"])
if EXCLUDE_MISSING_CUSTOMER_ID:
    analysis = analysis.dropna(subset=["customer_id"])

if EXCLUDE_NONPOSITIVE_QUANTITY_FROM_RFM:
    analysis = analysis[analysis["quantity"] > 0]
if EXCLUDE_NONPOSITIVE_PRICE_FROM_RFM:
    analysis = analysis[analysis["price"] > 0]

# Invoice IDs beginning with C are commonly used for cancellations.
# Inspect these records and decide whether to exclude them in your documented methodology.
analysis["invoice"] = analysis["invoice"].astype(str).str.strip()
analysis["is_cancellation_invoice"] = analysis["invoice"].str.upper().str.startswith("C")

# Explicit project assumption: exclude cancellation invoices from purchase RFM.
analysis = analysis[~analysis["is_cancellation_invoice"]].copy()

analysis["sales"] = analysis["quantity"] * analysis["price"]

print("Rows retained for RFM:", len(analysis))
print("Customers:", analysis["customer_id"].nunique())
print("Date range:", analysis["invoice_date"].min(), "to", analysis["invoice_date"].max())


## 4. Calculate Recency, Frequency and Monetary

In [ ]:
if ANALYSIS_DATE is None:
    analysis_date = analysis["invoice_date"].max().normalize() + pd.Timedelta(days=1)
else:
    analysis_date = pd.Timestamp(ANALYSIS_DATE).normalize()

rfm = (
    analysis.groupby("customer_id")
    .agg(
        last_purchase_date=("invoice_date", "max"),
        frequency=("invoice", "nunique"),
        monetary=("sales", "sum")
    )
    .reset_index()
)

rfm["recency"] = (analysis_date - rfm["last_purchase_date"].dt.normalize()).dt.days
rfm = rfm[["customer_id", "recency", "frequency", "monetary", "last_purchase_date"]]

display(rfm.head())
display(rfm[["recency", "frequency", "monetary"]].describe())


## 5. Score customers and assign business segments

This starter scoring uses quintiles. Recency is reversed: fewer days since the last purchase receives a higher score. Frequency and Monetary are scored in ascending order. Ties can cause fewer than five distinct score groups; `duplicates='drop'` handles this safely.

The segment rules below are an illustrative, transparent rule set. Review the cut-offs against the business context and document any changes before presenting results.

In [ ]:
def quintile_score(series, reverse=False):
    # Rank first so qcut can create bins even when many values are tied.
    ranks = series.rank(method="first", ascending=True)
    scores = pd.qcut(ranks, q=5, labels=False, duplicates="drop") + 1
    scores = scores.astype(int)
    return 6 - scores if reverse else scores

rfm["r_score"] = quintile_score(rfm["recency"], reverse=True)
rfm["f_score"] = quintile_score(rfm["frequency"])
rfm["m_score"] = quintile_score(rfm["monetary"])

rfm["rfm_score"] = (
    rfm["r_score"].astype(str)
    + rfm["f_score"].astype(str)
    + rfm["m_score"].astype(str)
)

def assign_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 3 and f >= 3:
        return "Loyal"
    if r <= 2 and (f >= 3 or m >= 3):
        return "At Risk"
    if r >= 4 and f <= 2:
        return "New"
    if r >= 3 and f <= 2:
        return "Needs Attention"
    return "Lost"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

display(rfm.head())
display(rfm["segment"].value_counts().to_frame("customer_count"))


## 6. Churn flag using the BRD's 90-day definition

In [ ]:
rfm["churned_flag"] = (rfm["recency"] > CHURN_DAYS).astype(int)

churn_summary = (
    rfm["churned_flag"]
    .value_counts()
    .rename(index={0: "Active", 1: "Churned"})
    .to_frame("customer_count")
)
churn_summary["percentage"] = (
    churn_summary["customer_count"] / len(rfm) * 100
)
display(churn_summary)

churned_percentage = rfm["churned_flag"].mean() * 100
print(f"Churned customer percentage: {churned_percentage:.2f}%")


## 7. Merge RFM results back into transaction data

In [ ]:
rfm_attributes = rfm[
    ["customer_id", "recency", "frequency", "monetary",
     "r_score", "f_score", "m_score", "rfm_score",
     "segment", "churned_flag"]
].copy()

# Enrich transaction-level rows with customer-level RFM attributes.
retail_rfm_final = analysis.merge(
    rfm_attributes,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Final rows, columns:", retail_rfm_final.shape)
display(retail_rfm_final.head())


## 8. Export Power BI-ready data

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
retail_rfm_final.to_csv(OUTPUT_FILE, index=False)

# Also export the customer-level RFM table for optional Power BI modelling.
rfm.to_csv(OUTPUT_FILE.parent / "customer_rfm_summary.csv", index=False)

print("Exported:", OUTPUT_FILE)
print("Exported:", OUTPUT_FILE.parent / "customer_rfm_summary.csv")


## 9. Exploratory summaries for dashboard planning

Use these summaries to guide the Power BI visuals. Verify all numbers after running the notebook.

In [ ]:
segment_summary = (
    retail_rfm_final.groupby("segment", dropna=False)
    .agg(
        customers=("customer_id", "nunique"),
        revenue=("sales", "sum"),
        transactions=("invoice", "nunique")
    )
    .sort_values("revenue", ascending=False)
)
display(segment_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=rfm, x="segment", y="monetary", estimator=np.median)
plt.title("Median Customer Monetary Value by Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Median Monetary Value")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 10. Methodology notes to complete

Before publishing the repository, record:

- The exact analysis reference date.
- Whether cancellation invoices and returns were excluded or treated separately.
- How missing Customer IDs were handled.
- How duplicate transactions were handled.
- RFM score binning method and segment rule thresholds.
- The exact definition of the Power BI Customer Value KPI.
- Actual row counts and segment / churn results after execution.

Do not report sample results or business impact until you have run and validated the analysis.